# NoCSA10 vs CSA10 — eight TEST5000 metrics
**CPU / Accelerator None; Internet ON.** No Secret, API token, Dataset, JPEG or checkpoint needed. Run once on either account.
Loads SHA-pinned512KB archived caption/reference snapshot from GitHub; same5000TEST IDs and25010references as latest PROMPT/CONTROL. No training, decoding or reranking.

Both exactXE10/beam5/max20. Compute BLEU1/METEOR/SPICE and replayBLEU4/ROUGE/CIDEr within1e-7; carry auditedCHAIRs/i only after snapshot SHA matches. Uses same COCO1.2/Java8/CoreNLP scorers as completed185/62. Output raw +×100 andCSV. NoCSA's lowerCHAIRs alone is not bettercaptionquality (many short sentences); encodercapacity differs.

4hCPU STOPcap, notETA; heartbeat30s and preservedoutput on any failure. Author checks only stdlib/metadata/mock/static, no actual Java/scorers. Send `plain_csa10_TEST_extra_v1_review.zip`.


In [ ]:
import os,sys,json,hashlib,urllib.request,subprocess,tempfile,time,threading,queue,signal,zipfile,traceback,shutil
from pathlib import Path
REVISION='c6306eb181361f031cdfdca52155669d9ff80a25'
HASHES={'score_prompt_test_extra.py': '8704765eb1c3c2713a7d31b92cba7454e905fcb3f0af37de627d81dc6d0f9bd7', 'score_plain_csa10_extra.py': 'b79c3128c3969ad4e594377c8f0c5d7463c34b4014ace8471722b7360f6caf52', 'test_plain_csa10_extra.py': '36175f18e2837e5458663ff23b0f644de69d608889607741ecb2ac992610c953', 'PLAIN_CSA10_EXTRA_INPUTS.json': '592e1b591e0732f819bf28cda5761c38c9758a3e0739f35ac32e004f58fe21df', 'plain_csa10_test_snapshot.json.gz': '440afb486d28e44066b6acc1d67ec4e26f9e4eb0b48419c491d7b6eef88bbfae'}
OUTPUT=Path('/kaggle/working/plain_csa10_TEST_extra_v1')
if OUTPUT.exists():raise RuntimeError('Preserve existing output; use fresh runtime/output path.')
if shutil.disk_usage('/kaggle/working').free<3*1024**3:raise RuntimeError('Need3GiB free for Java/CoreNLP/cache')
OUTPUT.mkdir(parents=True)
SOURCE=Path(tempfile.mkdtemp(prefix='plain-csa10-extra-',dir='/tmp'))
DEPS=SOURCE/'deps';LOG=OUTPUT/'notebook.log';DEADLINE=time.monotonic()+4*3600
for name,digest in HASHES.items():
    raw=urllib.request.urlopen('https://raw.githubusercontent.com/Supzxjee/Image_Captioning/'+REVISION+'/experiments/csa_paper/'+name,timeout=90).read()
    if hashlib.sha256(raw).hexdigest()!=digest:raise ValueError('Changed pinned source/snapshot:'+name)
    (SOURCE/name).write_bytes(raw)
(OUTPUT/'source_receipt.json').write_text(json.dumps(dict(commit=REVISION,sha256=HASHES),indent=2),encoding='utf-8')
ENV=os.environ.copy()
for key in ('KAGGLE_API_TOKEN','KAGGLE_USERNAME','KAGGLE_KEY'):ENV.pop(key,None)
ENV.update(PYTHONPATH=str(DEPS)+os.pathsep+str(SOURCE),PYTHONUNBUFFERED='1',OMP_NUM_THREADS='2',OPENBLAS_NUM_THREADS='2',CUDA_VISIBLE_DEVICES='')
def run(args,timeout=600,environment=None):
    remaining=DEADLINE-time.monotonic()
    if remaining<=0:raise TimeoutError('Parent CPU cap exceeded')
    timeout=min(timeout,remaining);start=time.monotonic();heartbeat=start;q=queue.Queue()
    args=list(map(str,args));print('Running:',' '.join(args),flush=True)
    process=subprocess.Popen(args,env=ENV if environment is None else environment,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,start_new_session=True)
    def reader():
        for line in process.stdout:q.put(line)
        q.put(None)
    thread=threading.Thread(target=reader,daemon=True);thread.start()
    try:
        with LOG.open('a',encoding='utf-8') as log:
            ended=False
            while not ended:
                try:
                    line=q.get(timeout=1)
                    if line is None:ended=True
                    else:print(line,end='',flush=True);log.write(line);log.flush()
                except queue.Empty:pass
                now=time.monotonic()
                if now-start>timeout:raise TimeoutError('CPU metric/phase cap exceeded; saved logs remain.')
                if now-heartbeat>=30:
                    print('CPU metrics active;',round(now-start),'s; no model inference.',flush=True);heartbeat=now
            if process.wait():raise subprocess.CalledProcessError(process.returncode,args)
    finally:
        if process.poll() is None:
            os.killpg(process.pid,signal.SIGTERM)
            try:process.wait(timeout=5)
            except subprocess.TimeoutExpired:os.killpg(process.pid,signal.SIGKILL);process.wait()


In [ ]:
complete=False
try:
    run([sys.executable,'-u',SOURCE/'test_plain_csa10_extra.py'],120)
    run([sys.executable,'-m','pip','install','-q','--no-deps','--target',DEPS,'pycocoevalcap==1.2'],600)
    for mode,cap in [('environment',2400),('smoke',900),('tokenize',600),('basic',1200),('meteor',2400),('spice',9000),('combine',120)]:
        print('CPU PHASE:',mode,flush=True)
        run([sys.executable,'-u',SOURCE/'score_plain_csa10_extra.py',mode,'--output',OUTPUT],cap)
    result=json.loads((OUTPUT/'result.json').read_text(encoding='utf-8'))
    complete=result['status']=='PLAIN_CSA10_TEST_EIGHT_METRICS_COMPLETE'
except BaseException:
    (OUTPUT/'failure.txt').write_text(traceback.format_exc(),encoding='utf-8');raise
finally:
    (OUTPUT/'run_status.json').write_text(json.dumps(dict(complete=complete,training=False,decoding=False,reranking=False)),encoding='utf-8')
    REVIEW=OUTPUT.with_name(OUTPUT.name+'_review.zip')
    with zipfile.ZipFile(REVIEW,'w',zipfile.ZIP_DEFLATED) as z:
        for p in OUTPUT.rglob('*'):
            if p.is_file() and 'assets' not in p.relative_to(OUTPUT).parts and p.suffix in ('.json','.csv','.txt','.log'):z.write(p,str(p.relative_to(OUTPUT)))
    print('SEND REVIEW:',REVIEW,'\nKEEP WHOLE OUTPUT:',OUTPUT,flush=True)
    print('COMPLETE: both baselines/8metrics/TEST5000' if complete else 'INCOMPLETE: preserve inputs/cache/logs; no inference or automatic retry.',flush=True)
